# MegaDetector Filtering

Runs MegaDetector v5a on all downloaded species images to generate bounding box annotations.
Images below the 0.5 confidence threshold are discarded as unusable.

Outputs detection JSONs to `MyDrive/bc-dashcam/detections/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install humanfriendly -q
!git clone https://github.com/agentmorris/MegaDetector.git
!pip install -e MegaDetector/ -q
!wget -q https://github.com/agentmorris/MegaDetector/releases/download/v5.0/md_v5a.0.0.pt -O MegaDetector/md_v5a.0.0.pt
print("Ready.")

## Run MegaDetector on each species

Change the species folder path for each run. Threshold is set low (0.1) to capture all possible detections; filtering at 0.5 happens in analysis.

In [ ]:
!python MegaDetector/megadetector/detection/run_detector_batch.py \
    MegaDetector/md_v5a.0.0.pt \
    /content/drive/MyDrive/bc-dashcam/data/moose \
    /content/drive/MyDrive/bc-dashcam/detections/moose_detections.json \
    --threshold 0.1 \
    --recursive

In [ ]:
!python MegaDetector/megadetector/detection/run_detector_batch.py \
    MegaDetector/md_v5a.0.0.pt \
    /content/drive/MyDrive/bc-dashcam/data/elk \
    /content/drive/MyDrive/bc-dashcam/detections/elk_detections.json \
    --threshold 0.1 \
    --recursive

In [ ]:
!python MegaDetector/megadetector/detection/run_detector_batch.py \
    MegaDetector/md_v5a.0.0.pt \
    /content/drive/MyDrive/bc-dashcam/data/bear \
    /content/drive/MyDrive/bc-dashcam/detections/bear_detections.json \
    --threshold 0.1 \
    --recursive

In [ ]:
!python MegaDetector/megadetector/detection/run_detector_batch.py \
    MegaDetector/md_v5a.0.0.pt \
    /content/drive/MyDrive/bc-dashcam/data/cougar \
    /content/drive/MyDrive/bc-dashcam/detections/cougar_detections.json \
    --threshold 0.1 \
    --recursive

In [ ]:
!python MegaDetector/megadetector/detection/run_detector_batch.py \
    MegaDetector/md_v5a.0.0.pt \
    /content/drive/MyDrive/bc-dashcam/data/deer \
    /content/drive/MyDrive/bc-dashcam/detections/deer_detections.json \
    --threshold 0.1 \
    --recursive

In [ ]:
!python MegaDetector/megadetector/detection/run_detector_batch.py \
    MegaDetector/md_v5a.0.0.pt \
    /content/drive/MyDrive/bc-dashcam/data/coyote \
    /content/drive/MyDrive/bc-dashcam/detections/coyote_detections.json \
    --threshold 0.1 \
    --recursive

In [ ]:
!python MegaDetector/megadetector/detection/run_detector_batch.py \
    MegaDetector/md_v5a.0.0.pt \
    /content/drive/MyDrive/bc-dashcam/data/raccoon \
    /content/drive/MyDrive/bc-dashcam/detections/raccoon_detections.json \
    --threshold 0.1 \
    --recursive

## Analyze detection results across all species

In [ ]:
import json
import os

species_list = ['moose', 'elk', 'bear', 'cougar', 'deer', 'coyote', 'raccoon']
detections_path = "/content/drive/MyDrive/bc-dashcam/detections"

print(f"{'Species':<12} {'Total':>8} {'Usable (>=0.5)':>15} {'Empty':>8} {'Low conf':>10} {'Usable %':>10}")
print("-" * 65)

for species in species_list:
    json_path = f"{detections_path}/{species}_detections.json"
    if not os.path.exists(json_path):
        print(f"{species:<12} FILE NOT FOUND")
        continue

    with open(json_path) as f:
        data = json.load(f)

    total = len(data['images'])
    usable = 0
    empty = 0
    low_conf = 0

    for img in data['images']:
        dets = img.get('detections', [])
        animal_found = any(d['category'] == '1' and d['conf'] >= 0.5 for d in dets)
        if animal_found:
            usable += 1
        elif not dets:
            empty += 1
        else:
            low_conf += 1

    pct = (usable / total * 100) if total > 0 else 0
    print(f"{species:<12} {total:>8} {usable:>15} {empty:>8} {low_conf:>10} {pct:>9.1f}%")

## Visualize sample detections

In [ ]:
import json
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import cv2

with open("/content/drive/MyDrive/bc-dashcam/detections/moose_detections.json") as f:
    detections = json.load(f)

good = [img for img in detections['images']
        if any(d['conf'] >= 0.5 and d['category'] == '1'
               for d in img.get('detections', []))]

print(f"Images with confident animal detections: {len(good)}")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, img_data in enumerate(good[:3]):
    img = np.array(Image.open(img_data['file']))
    h, w = img.shape[:2]

    for det in img_data['detections']:
        if det['conf'] >= 0.5 and det['category'] == '1':
            x, y, bw, bh = det['bbox']
            x1, y1 = int(x * w), int(y * h)
            x2, y2 = int((x + bw) * w), int((y + bh) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
            label = f"conf {det['conf']:.2f}"
            cv2.putText(img, label, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(f"Confidence: {max(d['conf'] for d in img_data['detections']):.2f}")

plt.tight_layout()
plt.show()